# 문제 2-4 : StructuredOutputParser 활용 - 여행 계획 분석기

여행 후기나 계획 텍스트에서 여행지, 기간, 예산, 추천도(1-5점), 주요 활동 리스트를 추출합니다.

- `StructuredOutputParser`와 `ResponseSchema` 사용
- 5개의 필드를 정의하여 정보 추출
- 자연어 텍스트에서 핵심 정보 파싱

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("OpenAI Key:", "OK" if OPENAI_API_KEY else "없음", "| Groq Key:", "OK" if GROQ_API_KEY else "없음")

### LLM 제공자 선택

`PROVIDER` 값만 `"groq"` 또는 `"openai"`로 바꾸면 이후 모든 셀의 모델이 바뀝니다.
Groq는 OpenAI 호환 API라서 둘 다 `ChatOpenAI` 클래스를 사용하고 `base_url`만 다릅니다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

llm = get_llm(temperature=0)

## 2. ResponseSchema 정의

- `ResponseSchema(name, description, type)` 목록이 곧 출력 스키마이고, 결과는 `name`을 키로 하는 **dict**입니다.
- `type`(기본값 `"string"`)은 **형식 지침에만 반영**될 뿐 값의 타입을 검증하지 않습니다. 그래서 `rating`은 문자열 `"4"`로 받습니다.
- 응답에 정의한 키가 빠지면 `OutputParserException`이 발생합니다.
- 타입 검증까지 필요하면 문제 2-3의 `PydanticOutputParser`를 사용합니다.

In [ ]:
response_schemas = [
    ResponseSchema(name="destination", description="여행지 (도시 또는 지역 이름)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="총 예산 또는 지출 금액 (예: 30만원). 언급이 없으면 '정보 없음'"),
    ResponseSchema(name="rating", description="5점 만점 추천도. 1~5 사이 숫자 하나. 언급이 없으면 글의 만족도로 판단"),
    ResponseSchema(name="activities", description="주요 활동 목록. 각 항목은 '장소 + 활동' 형태 (예: 해운대 바다구경)", type="List[string]"),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

## 3. 프롬프트 + LCEL 체인

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 후기·계획에서 핵심 정보를 추출하는 분석가입니다. 글에 있는 내용만 사용하세요.\n{format_instructions}"),
    ("user", "{text}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | output_parser

## 4. 실행

In [ ]:
text = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

result = chain.invoke({"text": text})
print(type(result))
print(json.dumps(result, indent=4, ensure_ascii=False))

### 4-1. 필드 활용

`rating`은 문자열이므로 계산에 쓰려면 직접 변환합니다.

In [ ]:
rating = int(result["rating"])
print(f"여행지: {result['destination']} ({result['duration']}, {result['budget']})")
print(f"추천도: {'★' * rating}{'☆' * (5 - rating)}")
for i, activity in enumerate(result["activities"], 1):
    print(f"  {i}. {activity}")

In [ ]:
texts = [
    "다음 달에 제주도 3박 4일 일정으로 갈 계획이에요. 예산은 50만원 잡았고, 한라산 등반이랑 우도 자전거 투어, 흑돼지 맛집 탐방을 하려고 해요. 기대 만점이라 5점!",
    "강릉 당일치기로 다녀왔는데 커피거리에서 커피 마시고 경포대 산책했어요. 사람이 너무 많아서 좀 지쳤네요. 3점 정도?",
]

for t in texts:
    try:
        print(json.dumps(chain.invoke({"text": t}), indent=4, ensure_ascii=False))
    except OutputParserException as e:
        print(f"파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)